In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from utils.shared import filter_uningested_data, write_to_table, notebook_exit
from utils.bronze import (
    create_bronze_weather_df,
    get_weather_zone,
    get_weather_zone_data,
)
from schema.bronze import weather_api_schema
from datetime import datetime
from logger.custom_logging import set_up_logger, get_job_logger
from utils.observability import create_control_table, insert_control_record
from pyspark.sql.functions import max
import logging


try:
    context = REPLACED_WITH_SPARK_CONF
    run_id = context.tags().get("runId").get()
except:
    # Fallback for when running as a file (not a notebook)
    run_id = "unknown"

log_info = {
    "layer": "bronze",
    "job": "extract_weather_data",
    "dataset": "openweathermap_api",
}

logger = set_up_logger()

log = get_job_logger(logger, **log_info, run_id=run_id)

log(logging.INFO, "Starting extract_weather_data")

# The variables below are for the control table which is used to track the last processed timestamp and observe run metrics

start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
pipeline = "project_voltstream"
layer = f"{log_info["layer"]}.{log_info["job"]}"
error_message = None
records_processed = 0
records_failed = 0
last_processed_timestamp = None

try:
    control_table = create_control_table(spark)
    df = spark.table("silver_dev.electrovolt.ev_stations")
    weather_zone = get_weather_zone(df, **log_info)

    api_key = "a7c31cb8512885e547d661cb539bb7bc"

    all_zones = get_weather_zone_data(weather_zone, api_key, **log_info)
    df = spark.createDataFrame(all_zones, schema=weather_api_schema)
    df = create_bronze_weather_df(df)
    write_to_table(df, "bronze_dev.electrovolt.weather_data", "overwrite", **log_info)
    status = "success"
    last_processed_timestamp = df.agg(max("ingest_timestamp")).collect()[0][0]
    records_processed = df.count()
    log(logging.INFO, "Finished extract_weather_data")

except Exception as e:
    status = "failed"
    error_message = str(e)
    raise

end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
insert_control_record(
    spark,
    control_table,
    pipeline,
    layer,
    last_processed_timestamp,
    records_processed,
    records_failed,
    start_time,
    end_time,
    error_message,
    status,
)
notebook_exit("SUCCESS")